# Eclat demo on a local Julia kernel

Open this notebook from a checkout of the repository in an IDE with a Julia kernel. The setup cell finds the checkout, and the next cell activates the project and installs its dependencies, so no clone or download of Julia is needed. Use Julia 1.13.2 to reproduce the recorded results; any 1.x from 1.12 on runs the cells.

The same steps run on Google Colab in `demo_colab.ipynb`, which differs only in setup: it clones the repository at an exact commit, installs Julia and Java, and draws the chart.

The notebook runs the package tests, mines a four-basket file worked out by hand, then mines the Mushroom benchmark with both variants and compares them with an independent SPMF run at five thresholds. The SPMF step needs Java 25 and the JAR from `tools/spmf/README.md`. The applications run in `market_basket_local.ipynb` and `image_patterns_local.ipynb`.

In [ ]:
# The notebook sits in notebooks/, so the checkout is its parent unless the working directory already is the checkout.
ROOT = abspath(get(ENV, "ITEMSET_ROOT", isfile("Project.toml") ? pwd() : dirname(pwd())))
isfile(joinpath(ROOT, "Project.toml")) ||
    error("Open the notebook from the repository root or from notebooks/, or set ITEMSET_ROOT to the checkout")
cd(ROOT)
println("Repository: ", ROOT)
println("Revision: ", ispath(".git") ? readchomp(`git rev-parse HEAD`) : "no .git folder; record the commit of this copy")


In [ ]:
using Pkg
v"1.12" <= VERSION < v"2" ||
    error("Use Julia 1.12 or later within 1.x; 1.13.2 gives the recorded results")
Pkg.activate(ROOT)
Pkg.instantiate()

using ItemsetMining
using Random
println("Julia: ", VERSION, " | threads: ", Threads.nthreads(), " | seed: ", DEFAULT_SEED)
println("Project: ", Base.active_project())
artifact_manifest = get(ENV, "ITEMSET_ARTIFACT_MANIFEST", "")
notebook_artifacts = Dict{String,String}()


In [ ]:
# Separate modules keep the definitions of the setup check and the tests out of the notebook workspace.
Base.include(Module(:ItemsetSetupCheck), joinpath(ROOT, "scripts/check_setup.jl"))
module ItemsetSetupTests
    include(joinpath(Main.ROOT, "test/runtests.jl"))
end

# Every later step runs a repository command in its own process, the way a user would. The helper prints that
# command for a terminal first. The kernel shows only what Julia itself prints, so the helper then collects the
# child's output and prints it, also when the command fails.
function repo(args...)
    println("\$ ", Base.shell_escape("julia", "--startup-file=no", "--project=.", args...))
    output = IOBuffer()
    try
        run(pipeline(`$(Base.julia_cmd()) --startup-file=no --project=$ROOT $args`; stdout=output, stderr=output))
    finally
        print(String(take!(output)))
    end
end

## A worked example

The input uses the SPMF format: one transaction per line, items as positive integers separated by spaces. `demo.jl` writes each frequent itemset with its support count as `items #SUP: count`. At minsup 0.5 an itemset must occur in 2 of the 4 baskets.

The expected file was worked out by hand. `tools/spmf/compare.jl` then compares the two complete maps, every itemset and every support, not only their counts.

In [ ]:
mkpath(joinpath(ROOT, "results/raw/demo-toy"))
println("Input, data/toy/baskets.txt:")
print(read(joinpath(ROOT, "data/toy/baskets.txt"), String))
repo("demo.jl", "--minsup", "0.5", "--output", "results/raw/demo-toy/baskets.txt", "data/toy/baskets.txt")

mined = readlines(joinpath(ROOT, "results/raw/demo-toy/baskets.txt"))
expected = readlines(joinpath(ROOT, "data/toy/baskets.expected.txt"))
println(rpad("mined by demo.jl", 20), "worked by hand")
for i in 1:max(length(mined), length(expected))
    println(rpad(get(mined, i, ""), 20), get(expected, i, ""))
end
println()
repo("tools/spmf/compare.jl", "data/toy/baskets.expected.txt", "results/raw/demo-toy/baskets.txt")

## Mushroom

Mushroom is a dense benchmark from the FIMI repository: 8,124 transactions over 119 items, 23 items each. The next cell downloads it, checks its SHA-256 against `data/benchmark/manifest.toml`, and prints its statistics. The cell after it mines it at minsup 0.3 with both variants and prints the longest frequent itemsets.

In [ ]:
repo("experiments/prepare_benchmarks.jl", "mushroom")

In [ ]:
mushroom = "data/benchmark/mushroom.txt"
repo("demo.jl", "--check-input", mushroom)
mkpath(joinpath(ROOT, "results/raw/demo-mushroom-mine"))
for variant in ("baseline", "optimized")
    repo("demo.jl", "--minsup", "0.3", "--variant", variant,
         "--output", "results/raw/demo-mushroom-mine/$variant.txt", mushroom)
end

outputs = Dict(variant => readlines(joinpath(ROOT, "results/raw/demo-mushroom-mine/$variant.txt"))
               for variant in ("baseline", "optimized"))
for (variant, lines) in sort!(collect(outputs))
    println(variant, ": ", length(lines), " frequent itemsets at minsup 0.3")
end
same = Set(outputs["baseline"]) == Set(outputs["optimized"])
println("Both variants return the same itemsets and supports: ", same)
same || error("The baseline and optimized outputs differ; compare them with tools/spmf/compare.jl")

longest = map(outputs["optimized"]) do line
    items, support = split(line, " #SUP: ")
    (; items, length=length(split(items)), support=parse(Int, support))
end
sort!(longest; by=row -> (row.length, row.support), rev=true)
println("\nlength\tsupport\titems")
foreach(row -> println(row.length, '\t', row.support, '\t', row.items), first(longest, 10))

## Comparison with SPMF

`experiments/run.jl` runs baseline, optimized and SPMF Eclat on Mushroom at five thresholds from `experiments/configs/demo-mushroom.toml`. Each time covers one process from spawn to exit, including Julia or JVM startup. After timing, the harness compares every output with a separate SPMF Eclat run as complete maps; `exact` with match rate 1.0 means every itemset and every support agree.

Put the pinned SPMF JAR at `tools/spmf/spmf.jar` using `tools/spmf/README.md`, and make Java 25 available on PATH or set `ITEMSET_JAVA` to its executable. One cold run per point shows the trend in a few minutes. The report's runtime figures use five repeats on a fixed host.

In [ ]:
module DemoExperiments
    include(joinpath(Main.ROOT, "experiments/Experiments.jl"))
end

java = get(ENV, "ITEMSET_JAVA", "java")
jar = joinpath(ROOT, "tools/spmf/spmf.jar")
if isfile(jar) && (isfile(java) || Sys.which(java) !== nothing)
    sweep = joinpath(ROOT, "results/raw/demo-mushroom")
    # run.jl refuses a non-empty output folder, so a rerun of this cell starts from an empty one.
    rm(sweep; force=true, recursive=true)
    repo("experiments/run.jl", "experiments/configs/demo-mushroom.toml",
         "--out", "results/raw/demo-mushroom", "--java", java)

    # SPMF attempts add a second row with SPMF's own timer; keep the spawn-to-exit row of every variant.
    rows = filter(row -> row["timing_scope"] == "process",
                  DemoExperiments.Experiments.read_csv_rows(joinpath(sweep, "runs.csv")))
    # run.jl records a failed attempt as a row and still exits 0, so a failure has to stop the notebook here.
    failed = filter(row -> row["status"] != "ok" || row["correctness_status"] != "exact", rows)
    foreach(row -> println(row["attempt_id"], ": ", row["status"], ", ", row["correctness_status"], ", ",
                           row["error"]), failed)
    isempty(failed) || error("$(length(failed)) sweep attempts failed or differ from SPMF; see $sweep/attempts")
    sort!(rows; by=row -> (-parse(Float64, row["minsup_relative"]), row["variant"]))
    columns = ("minsup_relative", "variant", "output_count", "status", "correctness_status", "match_rate",
               "elapsed_seconds")
    println(join(columns, '\t'))
    foreach(row -> println(join((row[column] for column in columns), '\t')), rows)
else
    println("The SPMF comparison needs Java 25 and tools/spmf/spmf.jar; tools/spmf/README.md shows how to set them up.")
end